In [ ]:

# Tesla Stock Price Prediction using SimpleRNN and LSTM

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import warnings

warnings.filterwarnings("ignore")

from google.colab import files
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense, Dropout


# Upload TSLA.csv file

uploaded = files.upload()


# Load Dataset

df = pd.read_csv("TSLA.csv")

display(df.head())

print(df.info())


# Data Cleaning

print("Missing Values")
print(df.isnull().sum())

df = df.fillna(method="ffill")
df = df.drop_duplicates()


# Date Processing

df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)


# EDA

plt.figure(figsize=(14,6))
plt.plot(df["Close"])
plt.title("Tesla Closing Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()


plt.figure(figsize=(14,6))
plt.plot(df["Open"], label="Open")
plt.plot(df["High"], label="High")
plt.plot(df["Low"], label="Low")
plt.legend()
plt.title("Tesla Stock Price Analysis")
plt.show()


plt.figure(figsize=(14,5))
plt.plot(df["Volume"])
plt.title("Tesla Volume Analysis")
plt.show()


# Feature Selection

data = df[["Close"]]


# Scaling

scaler = MinMaxScaler(feature_range=(0,1))

scaled_data = scaler.fit_transform(data)



# Create Time Series Dataset

def create_dataset(dataset,time_step=60):

    X=[]
    y=[]

    for i in range(time_step,len(dataset)):
        X.append(dataset[i-time_step:i,0])
        y.append(dataset[i,0])

    return np.array(X),np.array(y)


X,y = create_dataset(scaled_data)


X = X.reshape(
    X.shape[0],
    X.shape[1],
    1
)


# Train Test Split

train_size = int(len(X)*0.8)

X_train = X[:train_size]
X_test = X[train_size:]

y_train = y[:train_size]
y_test = y[train_size:]


# SimpleRNN Model

rnn_model = Sequential()

rnn_model.add(SimpleRNN(100,return_sequences=True,input_shape=(60,1)))

rnn_model.add(Dropout(0.2))

rnn_model.add(SimpleRNN(100))

rnn_model.add(Dense(1))

rnn_model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

rnn_model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2
)


# LSTM Model

lstm_model = Sequential()

lstm_model.add(LSTM(100,return_sequences=True,input_shape=(60,1)))

lstm_model.add(Dropout(0.2))

lstm_model.add(LSTM(100))

lstm_model.add(Dense(1))


lstm_model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)


lstm_model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2
)



# Predictions

rnn_pred = scaler.inverse_transform(
    rnn_model.predict(X_test)
)


lstm_pred = scaler.inverse_transform(
    lstm_model.predict(X_test)
)


actual = scaler.inverse_transform(
    y_test.reshape(-1,1)
)



# Graph Comparison

plt.figure(figsize=(15,7))

plt.plot(actual,label="Actual")

plt.plot(rnn_pred,label="SimpleRNN")

plt.plot(lstm_pred,label="LSTM")

plt.legend()

plt.title("Actual vs Prediction")

plt.show()



# Evaluation

rnn_rmse = math.sqrt(
    mean_squared_error(actual,rnn_pred)
)

lstm_rmse = math.sqrt(
    mean_squared_error(actual,lstm_pred)
)

print("SimpleRNN RMSE:",rnn_rmse)

print("LSTM RMSE:",lstm_rmse)



# Future Prediction

def predict_future(days,model):

    temp=list(scaled_data[-60:].reshape(-1))

    result=[]

    for i in range(days):

        x=np.array(temp[-60:]).reshape(1,60,1)

        pred=model.predict(x)

        temp.append(pred[0][0])

        result.append(pred[0][0])


    return scaler.inverse_transform(
        np.array(result).reshape(-1,1)
    )


print("1 Day Prediction")
print(predict_future(1,lstm_model))


print("5 Days Prediction")
print(predict_future(5,lstm_model))


print("10 Days Prediction")
print(predict_future(10,lstm_model))


# Save model for Streamlit

lstm_model.save("tesla_lstm_model.h5")

print("Project Completed Successfully")
